# Ten-year forest-notice exploration
This notebook reads only completed monthly artifacts from `data/processed/notices`.

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
processed_store = importlib.import_module('processed_notice_store')
load_processed_notices = processed_store.load_processed_notices
load_processed_species = processed_store.load_processed_species
PROCESSED_ROOT = ROOT / 'data' / 'processed' / 'notices'

In [ ]:
notices = load_processed_notices(PROCESSED_ROOT)
species = load_processed_species(PROCESSED_ROOT)
{'notices': notices.shape, 'species_rows': species.shape, 'crs': str(notices.crs)}

In [ ]:
notices.info()
notices.isna().mean().sort_values(ascending=False).head(20)

In [ ]:
date_values = []
for _, row in notices.iterrows():
    column = row.get('_date_col')
    date_values.append(row.get(column) if isinstance(column, str) else None)
notices = notices.assign(_notice_date=pd.to_datetime(date_values, errors='coerce', utc=True))
monthly = notices.groupby(notices['_notice_date'].dt.to_period('M')).agg(
    notices=('notice_ix', 'size'),
    area_ha=('area_ha', 'sum'),
    standing_biomass_tco2=('standing_live_biomass_tco2', 'sum'),
)
monthly.plot(subplots=True, figsize=(14, 9), title='Monthly processed notices')

In [ ]:
species_summary = species.groupby('species', as_index=False)['biomass_tco2'].sum()
species_summary.sort_values('biomass_tco2', ascending=False).head(20)

For spatial work, sample first: `sample = notices.sample(min(2000, len(notices)), random_state=42)`. Avoid rendering the complete geometry collection interactively.